# Allen Brain Cell Atlas — MERFISH Data Access Tutorial
## Step-by-step guide: Loading data, visualising sections, and analysing Mc4r co-expression

**Follow this notebook in order — each cell must be run before the next.**

---

## BEFORE YOU START — Prerequisites

You need:
- Python 3.9 or newer (Anaconda recommended)
- A stable internet connection (data downloads from AWS S3 — free, no account needed)
- ~5 GB free disk space for the metadata + one section's expression matrix

**Recommended:** Run this in Jupyter Lab or Jupyter Notebook.
To open Jupyter: open a terminal/Anaconda prompt and type `jupyter lab`

---

In [2]:
import sys
print(sys.executable)
print(sys.version)

/rds/user/mmw72/hpc-work/envs/zarr3_uv/bin/python3
3.11.13 (main, Apr 28 2026, 13:06:52) [GCC 8.5.0 20210514 (Red Hat 8.5.0-28)]


In [10]:
!du -sh ~/* ~/.* 2>/dev/null | sort -rh | head -20

^C


In [12]:
!export PATH="/rds/user/mmw72/hpc-work/uv-bin:$PATH"
!which uv
uv --version

/usr/bin/which: no uv in (/rds/user/mmw72/hpc-work/envs/zarr3_uv/bin:/usr/local/software/archive/linux-scientific7-x86_64/gcc-9/miniconda3-4.7.12.1-rmuek6r3f6p3v6fdj7o2klyzta3qhslh/bin:/usr/local/software/archive/linux-scientific7-x86_64/gcc-9/miniconda3-4.7.12.1-rmuek6r3f6p3v6fdj7o2klyzta3qhslh/condabin:/usr/local/software/spack/spack-views/rocky8-icelake-20220710/intel-oneapi-mpi-2021.6.0/intel-2021.6.0/guxuvcpmykplbrr2e3af2yd7njqhau5e/mpi/2021.6.0/libfabric/bin:/usr/local/software/spack/spack-views/rocky8-icelake-20220710/intel-oneapi-mpi-2021.6.0/intel-2021.6.0/guxuvcpmykplbrr2e3af2yd7njqhau5e/mpi/2021.6.0/bin:/usr/local/software/spack/csd3/spack-views/ucx-2024-08-19/ucx-1.15.0/gcc-8.5.0/3odtpik4wnhzdj6fgnc5ujwcmmnx4yjl/bin:/usr/local/software/spack/spack-views/rocky8-icelake-20220710/intel-oneapi-compilers-2022.1.0/gcc-11.3.0/b6zld2mz7cid27yloxznoidymd7vywwz/compiler/2022.1.0/linux/lib/oclfpga/bin:/usr/local/software/spack/spack-views/rocky8-icelake-20220710/intel-oneapi-compilers

NameError: name 'uv' is not defined

## STEP 1 — Install the Allen Brain Cell Atlas package

Run this once. It installs the `abc_atlas_access` library directly from GitHub.
This is the official library for accessing all the Allen MERFISH data.

In [13]:
# Run this cell once to install the package
# The exclamation mark runs it as a terminal command from inside Jupyter
import sys
!{sys.executable} -m pip install "abc_atlas_access[notebooks] @ git+https://github.com/alleninstitute/abc_atlas_access.git"

/rds/user/mmw72/hpc-work/envs/zarr3_uv/bin/python3: No module named pip


## STEP 2 — Import libraries

These are all the Python libraries we will use throughout this notebook.
If any fail to import, install them with `pip install <library_name>`.

In [14]:
import pandas as pd
from pathlib import Path
import numpy as np
import anndata          # for reading .h5ad expression matrix files
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

print("All imports successful!")

All imports successful!


## STEP 3 — Set up the data download folder

Choose a folder on your computer where all the data will be saved.
Change the path below to wherever you want it — it will be created automatically.

> **Note:** The full dataset is very large (~hundreds of GB). We will download only what we need.

In [15]:
# ---- CHANGE THIS PATH to wherever you want your data saved ----
download_base = Path('/rds/user/mmw72/hpc-work/abc_atlas_data')# Windows example
# download_base = Path('/Users/YourName/abc_atlas_data')   # Mac example
# download_base = Path('/home/yourname/abc_atlas_data')    # Linux example
# ---------------------------------------------------------------

# Create the folder if it doesn't exist
download_base.mkdir(parents=True, exist_ok=True)

# Connect to the Allen Institute's AWS data store
# This downloads only a small manifest file (~KB) listing all available data
abc_cache = AbcProjectCache.from_s3_cache(download_base)

# Check which manifest version is loaded (should show something like 'releases/20250531/manifest.json')
print("Connected! Current manifest:", abc_cache.current_manifest)

Connected! Current manifest: releases/20260415/manifest.json


## STEP 4 — Download and load cell metadata

This downloads a table with one row per cell (~4 million cells).
Each row contains: brain section label, x/y/z coordinates, cell type cluster assignment.

**File size: ~300 MB. Downloads once, then loads from disk.**

In [16]:
# Download and load cell metadata
# First time: downloads from S3. Subsequent times: loads from local disk.
cell = abc_cache.get_metadata_dataframe(
    directory='MERFISH-C57BL6J-638850',
    file_name='cell_metadata',
    dtype={"cell_label": str}
)
cell.set_index('cell_label', inplace=True)

print(f"Total number of cells in dataset: {len(cell):,}")
print(f"\nColumns available:")
for col in cell.columns:
    print(f"  - {col}")
print(f"\nFirst 3 rows:")
cell.head(3)

Total number of cells in dataset: 3,938,808

Columns available:
  - brain_section_label
  - cluster_alias
  - average_correlation_score
  - feature_matrix_label
  - donor_label
  - donor_genotype
  - donor_sex
  - x
  - y
  - z
  - abc_sample_id

First 3 rows:


,brain_section_label,cluster_alias,average_correlation_score,feature_matrix_label,donor_label,donor_genotype,donor_sex,x,y,z,abc_sample_id
cell_label,,,,,,,,,,,
1019171907102340387-1,C57BL6J-638850.37,1408,0.596276,C57BL6J-638850,C57BL6J-638850,wt/wt,M,7.226245,4.148963,6.6,c9881423-76a7-4835-ba8b-7942fd384b6b
1104095349101460194-1,C57BL6J-638850.26,4218,0.641180,C57BL6J-638850,C57BL6J-638850,wt/wt,M,5.064889,7.309543,4.2,aa815488-6487-4e47-8a5e-d82ac9933bc6
1017092617101450577,C57BL6J-638850.25,4218,0.763531,C57BL6J-638850,C57BL6J-638850,wt/wt,M,5.792921,8.189973,4.0,91ef7a85-8e3e-4410-8ee2-785788df3ebe


## STEP 5 — Load cell type annotation labels

This connects each cell's `cluster_alias` number to a human-readable name
(e.g. class, subclass, supertype, cluster).

In [17]:
# Download cluster annotation table
cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
)
cluster_details.set_index('cluster_alias', inplace=True)

# Download colour table (useful for plotting)
cluster_colors = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_color'
)
cluster_colors.set_index('cluster_alias', inplace=True)

print("Cluster annotation columns:", list(cluster_details.columns))
cluster_details.head(3)

Cluster annotation columns: ['neurotransmitter', 'class', 'subclass', 'supertype', 'cluster']


,neurotransmitter,class,subclass,supertype,cluster
cluster_alias,,,,,
1,Glut,01 IT-ET Glut,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3
2,Glut,01 IT-ET Glut,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0327 L2 IT PPP-APr Glut_3
3,Glut,01 IT-ET Glut,018 L2 IT PPP-APr Glut,0081 L2 IT PPP-APr Glut_2,0322 L2 IT PPP-APr Glut_2


## STEP 6 — Merge cell metadata with cluster annotations

This gives every cell a full cell type label alongside its spatial coordinates.

In [18]:
# Join the cell table with the annotation and colour tables
cell_extended = cell.join(cluster_details, on='cluster_alias')
cell_extended = cell_extended.join(cluster_colors, on='cluster_alias')

print(f"Cells with full annotation: {len(cell_extended):,}")
print(f"\nExample — unique cell classes in dataset:")
print(sorted(cell_extended['class'].unique())[:10], "...")

Cells with full annotation: 3,938,808

Example — unique cell classes in dataset:
['01 IT-ET Glut', '02 NP-CT-L6b Glut', '03 OB-CR Glut', '04 DG-IMN Glut', '05 OB-IMN GABA', '06 CTX-CGE GABA', '07 CTX-MGE GABA', '08 CNU-MGE GABA', '09 CNU-LGE GABA', '10 LSX GABA'] ...


## STEP 7 — Look at what brain sections are available

There are 59 coronal sections from anterior to posterior.
We'll list them so you can choose which section to focus on.

In [19]:
sections = sorted(cell_extended['brain_section_label'].unique())
print(f"Total sections: {len(sections)}")
print("\nAll section labels (sorted anterior to posterior):")
for i, s in enumerate(sections):
    print(f"  [{i:02d}] {s}")

Total sections: 59

All section labels (sorted anterior to posterior):
  [00] C57BL6J-638850.01
  [01] C57BL6J-638850.02
  [02] C57BL6J-638850.03
  [03] C57BL6J-638850.04
  [04] C57BL6J-638850.05
  [05] C57BL6J-638850.06
  [06] C57BL6J-638850.08
  [07] C57BL6J-638850.09
  [08] C57BL6J-638850.10
  [09] C57BL6J-638850.11
  [10] C57BL6J-638850.12
  [11] C57BL6J-638850.13
  [12] C57BL6J-638850.14
  [13] C57BL6J-638850.15
  [14] C57BL6J-638850.16
  [15] C57BL6J-638850.17
  [16] C57BL6J-638850.18
  [17] C57BL6J-638850.19
  [18] C57BL6J-638850.24
  [19] C57BL6J-638850.25
  [20] C57BL6J-638850.26
  [21] C57BL6J-638850.27
  [22] C57BL6J-638850.28
  [23] C57BL6J-638850.29
  [24] C57BL6J-638850.30
  [25] C57BL6J-638850.31
  [26] C57BL6J-638850.32
  [27] C57BL6J-638850.33
  [28] C57BL6J-638850.35
  [29] C57BL6J-638850.36
  [30] C57BL6J-638850.37
  [31] C57BL6J-638850.38
  [32] C57BL6J-638850.39
  [33] C57BL6J-638850.40
  [34] C57BL6J-638850.42
  [35] C57BL6J-638850.43
  [36] C57BL6J-638850.44
  [3

## STEP 8 — Download the gene expression matrix (raw counts)

This is the big file — it contains RNA counts for all 500 genes across all ~4 million cells.

> **⚠️ File size: ~2-3 GB. This will take several minutes to download the first time.**
> After that it loads from disk in seconds.

The data is in **AnnData h5ad format** — a standard single-cell genomics format.
- `raw` = raw transcript counts (integers)
- `log2` = log2-normalised counts (better for visualisation)

In [20]:
# This downloads and opens the RAW COUNTS expression matrix
# backed=True means it doesn't load everything into RAM at once
print("Downloading expression matrix (this may take a few minutes first time)...")

expression_file = abc_cache.get_file_path(
    directory='MERFISH-C57BL6J-638850',
    file_name='C57BL6J-638850/raw'
)

adata = anndata.read_h5ad(expression_file, backed='r')

print(f"Expression matrix loaded!")
print(f"  Shape: {adata.n_obs:,} cells × {adata.n_vars} genes")
print(f"  Gene names (first 10): {list(adata.var_names[:10])}")

Expression matrix loaded!
  Shape: 4,334,174 cells × 550 genes
  Gene names (first 10): ['ENSMUSG00000026778', 'ENSMUSG00000026837', 'ENSMUSG00000001985', 'ENSMUSG00000039323', 'ENSMUSG00000048387', 'ENSMUSG00000027849', 'ENSMUSG00000033063', 'ENSMUSG00000030226', 'ENSMUSG00000020902', 'ENSMUSG00000021685']


## STEP 9 — Find Mc4r in the gene panel

Check that Mc4r is in the 500-gene panel and get its identifier.

In [21]:
# Load gene metadata
gene = abc_cache.get_metadata_dataframe(
    directory='MERFISH-C57BL6J-638850',
    file_name='gene'
)
gene.set_index('gene_identifier', inplace=True)

print(f"Total genes in panel: {len(gene)}")
print("\nSearching for Mc4r...")

# Search for Mc4r (case-insensitive)
mc4r_hits = gene[gene['gene_symbol'].str.lower() == 'mc4r']
print(mc4r_hits)

# Also search for similar names just in case
print("\nAll genes containing 'mc4' or 'Mc4':")
print(gene[gene['gene_symbol'].str.contains('Mc4', case=False)])

Total genes in panel: 550

Searching for Mc4r...
Empty DataFrame
Columns: [gene_symbol, transcript_identifier, name, mapped_ncbi_identifier]
Index: []

All genes containing 'mc4' or 'Mc4':
Empty DataFrame
Columns: [gene_symbol, transcript_identifier, name, mapped_ncbi_identifier]
Index: []


## STEP 10 — Extract Mc4r expression for all cells

Pull out just the Mc4r counts column from the huge expression matrix.
This is memory-efficient — we only load one gene at a time.

In [ ]:
# Define the gene(s) you want to extract
# Mc4r plus every marker gene you want a co-expression slide for
MARKER_GENES = ['Gpr83', 'Glp1r', 'Qrfpr', 'Mc3r', 'Sim1']   # <-- YOUR MARKER GENES
GENES_OF_INTEREST = ['Mc4r'] + MARKER_GENES

# Find the Ensembl identifiers for these genes
gene_ids = gene[gene['gene_symbol'].isin(GENES_OF_INTEREST)].index.tolist()
gene_symbols = gene.loc[gene_ids, 'gene_symbol'].tolist()

missing = sorted(set(GENES_OF_INTEREST) - set(gene_symbols))
if missing:
    print(f"WARNING: these genes are NOT in the 500-gene MERFISH panel and will be skipped: {missing}")
    print("(You'll need the imputed gene dataset — see the Summary section at the end — for these.)")

print(f"\nFound gene IDs: {list(zip(gene_symbols, gene_ids))}")

# Extract expression values for just these genes
# This slices the large matrix efficiently
print("\nExtracting expression values...")
expr_subset = pd.DataFrame(
    adata[:, gene_ids].X.toarray(),    # convert sparse matrix to dense
    index=adata.obs_names,
    columns=gene_symbols
)

print(f"Extracted shape: {expr_subset.shape}")
print(f"\nExpression summary (cells with >0 transcripts):")
for g in gene_symbols:
    print(f"  {g}: {(expr_subset[g] > 0).sum():,} cells")

(You'll need the imputed gene dataset — see the Summary section at the end — for these.)

Found gene IDs: [('Gpr83', 'ENSMUSG00000031932'), ('Qrfpr', 'ENSMUSG00000058400')]

Extracting expression values...
Extracted shape: (4334174, 2)

Expression summary (cells with >0 transcripts):
  Gpr83: 1,331,088 cells
  Qrfpr: 367,250 cells


## STEP 11 — Join expression data with spatial coordinates

Combine the expression values with the cell's spatial position and annotations.

In [ ]:
# Join expression with cell metadata
cell_with_expr = cell_extended.join(expr_subset)

print(f"Combined table shape: {cell_with_expr.shape}")
print(f"Columns now include: {list(cell_with_expr.columns[-6:])}")
cell_with_expr[['x', 'y', 'z', 'class', 'subclass', 'Mc4r']].head(5)

Combined table shape: (3938808, 23)
Columns now include: ['class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'Gpr83', 'Qrfpr']


KeyError: "['Mc4r'] not in index"

## STEP 12 — Plot Mc4r expression across one brain section

Visualise where Mc4r-expressing cells are located in a single coronal section.

**Choose a section number from the list in Step 7.**
Sections around index 25-35 typically contain hypothalamus (where Mc4r is enriched).

In [ ]:
# ---- CHOOSE YOUR SECTION HERE ----
SECTION_LABEL = sections[30]   # Change the index number to explore different sections
# ----------------------------------

print(f"Plotting section: {SECTION_LABEL}")

# Filter to this section
sec = cell_with_expr[cell_with_expr['brain_section_label'] == SECTION_LABEL]
print(f"Cells in this section: {len(sec):,}")

# Create the plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f'Section: {SECTION_LABEL}', fontsize=14)

# LEFT PANEL: all cells coloured by class
ax = axes[0]
ax.set_title('All cells — coloured by cell class', fontsize=11)
scatter = ax.scatter(
    sec['x'], sec['y'],
    c=sec['class_color'],   # colour from Allen annotation
    s=0.5, alpha=0.6
)
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_aspect('equal')
ax.invert_yaxis()

# RIGHT PANEL: cells coloured by Mc4r expression level
ax = axes[1]
ax.set_title('Mc4r expression (raw counts)', fontsize=11)

# Plot non-expressing cells as grey background
no_expr = sec[sec['Mc4r'] == 0]
ax.scatter(no_expr['x'], no_expr['y'], c='lightgrey', s=0.3, alpha=0.3, label='Mc4r = 0')

# Plot expressing cells coloured by count level
yes_expr = sec[sec['Mc4r'] > 0]
sc = ax.scatter(
    yes_expr['x'], yes_expr['y'],
    c=yes_expr['Mc4r'],
    cmap='hot_r',
    s=5, alpha=0.9,
    vmin=1, vmax=yes_expr['Mc4r'].quantile(0.99)
)
plt.colorbar(sc, ax=ax, label='Mc4r transcript count')
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_aspect('equal')
ax.invert_yaxis()

print(f"Mc4r+ cells in this section: {len(yes_expr):,}")
plt.tight_layout()
plt.savefig('Mc4r_section_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'Mc4r_section_plot.png'")

## STEP 13 — Find which sections have the most Mc4r+ cells

This scans all 59 sections to find where Mc4r expression is highest —
useful for targeting your analysis.

In [ ]:
# Count Mc4r+ cells per section (threshold: at least 1 transcript)
mc4r_per_section = (
    cell_with_expr[cell_with_expr['Mc4r'] > 0]
    .groupby('brain_section_label')
    .size()
    .sort_values(ascending=False)
)

print("Top 15 sections by number of Mc4r+ cells:")
print(mc4r_per_section.head(15).to_string())

# Bar chart of Mc4r+ cells per section
fig, ax = plt.subplots(figsize=(14, 4))
mc4r_by_section_ordered = (
    cell_with_expr[cell_with_expr['Mc4r'] > 0]
    .groupby('brain_section_label')
    .size()
    .reindex(sections, fill_value=0)
)
ax.bar(range(len(sections)), mc4r_by_section_ordered.values, color='crimson')
ax.set_xticks(range(len(sections)))
ax.set_xticklabels([s.split('.')[-1] for s in sections], rotation=90, fontsize=7)
ax.set_xlabel('Brain section (anterior → posterior)')
ax.set_ylabel('Number of Mc4r+ cells')
ax.set_title('Mc4r+ cells across all coronal sections')
plt.tight_layout()
plt.savefig('Mc4r_per_section.png', dpi=150, bbox_inches='tight')
plt.show()

## STEP 14 — Co-expression analysis: Mc4r with each marker gene

Now the key analysis for your project — for **each** marker gene in `MARKER_GENES`,
how many cells co-express it with Mc4r?

This gives you counts comparable to double-labelling immunofluorescence, for every
gene in one pass (no need to edit and re-run cell-by-cell).

In [ ]:
# Genes already extracted in Step 10: MARKER_GENES = ['Gpr83', 'Glp1r', 'Qrfpr', 'Mc3r', 'Sim1']

mc4r_pos = cell_with_expr['Mc4r'] > 0
n_total  = len(cell_with_expr)
n_mc4r   = mc4r_pos.sum()

coexpr_summary = {}   # store results per gene for later steps

for gene_name in MARKER_GENES:
    if gene_name not in cell_with_expr.columns:
        print(f"--- Skipping {gene_name}: not found in extracted expression data (see Step 10 warning) ---\n")
        continue

    coexpr_pos = cell_with_expr[gene_name] > 0
    n_coexpr   = coexpr_pos.sum()
    n_both     = (mc4r_pos & coexpr_pos).sum()
    n_mc4r_only   = (mc4r_pos & ~coexpr_pos).sum()
    n_coexpr_only = (~mc4r_pos & coexpr_pos).sum()

    coexpr_summary[gene_name] = {
        'n_total': n_total, 'n_mc4r': n_mc4r, 'n_coexpr': n_coexpr,
        'n_both': n_both, 'n_mc4r_only': n_mc4r_only, 'n_coexpr_only': n_coexpr_only,
    }

    print(f"=== WHOLE BRAIN Co-expression: Mc4r × {gene_name} ===")
    print(f"Total cells:                  {n_total:>10,}")
    print(f"Mc4r+ cells:                  {n_mc4r:>10,}  ({100*n_mc4r/n_total:.2f}%)")
    print(f"{gene_name}+ cells:           {n_coexpr:>10,}  ({100*n_coexpr/n_total:.2f}%)")
    print(f"Mc4r+ AND {gene_name}+:       {n_both:>10,}  ({100*n_both/n_mc4r:.1f}% of Mc4r+ cells)")
    print(f"Mc4r+ only:                   {n_mc4r_only:>10,}")
    print(f"{gene_name}+ only:            {n_coexpr_only:>10,}\n")

## STEP 15 (OPTIONAL) — Co-expression in a specific brain region

**You don't need to run this step** — Steps 14 and 16 already give you whole-section
co-expression slides. This step is here only if you later want to restrict the analysis
to a specific brain region/cell-type subclass (e.g. hypothalamus, arcuate nucleus) for a
more focused comparison to your own experimental data.

In [ ]:
# First, see what subclass labels are available (these map to brain regions)
print("All subclasses containing 'HY' (hypothalamus cells):")
hy_subclasses = sorted([
    s for s in cell_with_expr['subclass'].unique() 
    if 'HY' in str(s) or 'hyp' in str(s).lower() or 'Arc' in str(s)
])
for s in hy_subclasses:
    print(f"  {s}")

In [ ]:
# ---- FILTER TO A BRAIN REGION OF INTEREST (optional example — edit as needed) ----
REGION_FILTER = 'HY'        # Will match any subclass containing 'HY'
REGION_GENE    = 'Gpr83'    # Which marker gene to check co-expression for, within this region

region_cells = cell_with_expr[
    cell_with_expr['subclass'].str.contains(REGION_FILTER, na=False)
]

print(f"Cells in region matching '{REGION_FILTER}': {len(region_cells):,}")

# Co-expression within this region
mc4r_pos_r   = region_cells['Mc4r'] > 0
coexpr_pos_r = region_cells[REGION_GENE] > 0

n_total_r  = len(region_cells)
n_mc4r_r   = mc4r_pos_r.sum()
n_coexpr_r = coexpr_pos_r.sum()
n_both_r   = (mc4r_pos_r & coexpr_pos_r).sum()

print(f"\n=== {REGION_FILTER} REGION: Mc4r × {REGION_GENE} ===")
print(f"Total cells in region:            {n_total_r:>10,}")
print(f"Mc4r+ cells:                      {n_mc4r_r:>10,}  ({100*n_mc4r_r/n_total_r:.2f}%)")
print(f"{REGION_GENE}+ cells:             {n_coexpr_r:>10,}  ({100*n_coexpr_r/n_total_r:.2f}%)")
if n_mc4r_r > 0:
    print(f"Co-expressing cells:              {n_both_r:>10,}  ({100*n_both_r/n_mc4r_r:.1f}% of Mc4r+ cells)")

## STEP 16 — Generate one coronal-section slide per marker gene

For each gene in `MARKER_GENES`, this saves a **separate** figure (not a combined grid) —
ready to drop straight into a slide deck. All genes are plotted on the same coronal
section (the one with the most Mc4r+ cells, from Step 13) so the anatomy is directly
comparable across slides. Each plot shows the whole section with co-expressing cells
highlighted, with no restriction to a specific cluster.

Files are saved as `Mc4r_<gene>_coexpression.png`.

In [ ]:
# Use the top section for Mc4r from Step 13 — kept the same across all genes
# so the anatomy lines up when you compare slides side by side.
BEST_SECTION = mc4r_per_section.index[0]
print(f"Using section: {BEST_SECTION}\n")

sec2 = cell_with_expr[cell_with_expr['brain_section_label'] == BEST_SECTION].copy()

# Optional: cluster each marker is enriched in (for the slide subtitle).
# Edit/extend this from your own marker table — leave blank ('') if not known.
CLUSTER_LABELS = {
    'Gpr83': 'VMH Nr5a1 prdm12 Glut',
    'Glp1r': 'MPN-MPO Hmx2 Eomes Glut',
    'Qrfpr': 'VMH Fez1 Glut',
    'Mc3r':  'VMH Fez1 Glut',
    'Sim1':  '',
}

saved_files = []

for gene_name in MARKER_GENES:
    if gene_name not in sec2.columns:
        print(f"--- Skipping {gene_name}: not found in extracted expression data ---")
        continue

    # Assign category to each cell for this gene
    mc4r_pos_s   = sec2['Mc4r'] > 0
    gene_pos_s   = sec2[gene_name] > 0
    category = pd.Series('Neither', index=sec2.index)
    category[mc4r_pos_s & gene_pos_s]  = 'Both'
    category[mc4r_pos_s & ~gene_pos_s] = 'Mc4r only'
    category[~mc4r_pos_s & gene_pos_s] = f'{gene_name} only'
    sec2['category'] = category

    color_map = {
        'Neither':              ('lightgrey', 0.2, 0.5),
        f'{gene_name} only':    ('royalblue', 0.8, 4),
        'Mc4r only':             ('tomato',     0.8, 4),
        'Both':                  ('gold',       1.0, 8),
    }

    fig, ax = plt.subplots(figsize=(10, 9))
    cluster_label = CLUSTER_LABELS.get(gene_name, '')
    subtitle = f' ({cluster_label})' if cluster_label else ''
    ax.set_title(f'Co-expression: Mc4r × {gene_name}{subtitle}\nSection: {BEST_SECTION}', fontsize=12)

    # Plot in order: background first, co-expressing cells on top
    for cat in ['Neither', f'{gene_name} only', 'Mc4r only', 'Both']:
        subset = sec2[sec2['category'] == cat]
        color, alpha, size = color_map[cat]
        ax.scatter(subset['x'], subset['y'],
                   c=color, s=size, alpha=alpha, label=f'{cat} (n={len(subset):,})')

    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    plt.tight_layout()

    out_file = f'Mc4r_{gene_name}_coexpression.png'
    plt.savefig(out_file, dpi=150, bbox_inches='tight')
    plt.show()
    saved_files.append(out_file)
    print(f"Saved '{out_file}'\n")

print(f"\nAll done — {len(saved_files)} individual slide(s) saved:")
for f in saved_files:
    print(f"  {f}")

## STEP 17 — Export results as CSV for your records

Save the co-expression data for all Mc4r+ cells as a spreadsheet.

In [ ]:
# Export Mc4r+ cells with all metadata and expression
mc4r_positive_cells = cell_with_expr[cell_with_expr['Mc4r'] > 0].copy()

# Select useful columns for export
found_marker_genes = [g for g in MARKER_GENES if g in mc4r_positive_cells.columns]
export_cols = [
    'brain_section_label', 'x', 'y', 'z',
    'class', 'subclass', 'supertype', 'cluster',
    'neurotransmitter', 'Mc4r'
] + found_marker_genes

export_df = mc4r_positive_cells[[c for c in export_cols if c in mc4r_positive_cells.columns]]

output_file = 'Mc4r_positive_cells.csv'
export_df.to_csv(output_file)

print(f"Exported {len(export_df):,} Mc4r+ cells to '{output_file}'")
print(f"\nSummary of exported data:")
export_df.head()

---
## Summary & Next Steps

You now have:
- ✅ Cell metadata with spatial coordinates for ~4 million cells
- ✅ Raw transcript counts for Mc4r and all your marker genes
- ✅ Mc4r+ cell locations across all 59 brain sections
- ✅ Whole-brain co-expression counts (Mc4r × each marker gene)
- ✅ One standalone coronal-section PNG per marker gene — ready to drop into slides
- ✅ CSV export for comparison with your own experimental data

**To extend this analysis:**

1. **Add or remove marker genes:** Edit `MARKER_GENES` in Step 10 — that one list drives extraction, the Step 14 stats, and the Step 16 slides.

2. **Genes NOT in the 500-gene panel** (including Mc4r if it isn't found): Use the **imputed gene dataset** — this extends coverage to ~8,000 genes. Tutorial: `alleninstitute.github.io/abc_atlas_access/notebooks/merfish_imputed_genes_example.html`

3. **Use the Zhuang dataset** (which is what you have open in the browser — Zhuang-ABCA-1): Replace `'MERFISH-C57BL6J-638850'` with `'Zhuang-ABCA-1'` in the cache calls. Tutorial: `alleninstitute.github.io/abc_atlas_access/notebooks/zhuang_merfish_tutorial.html`

4. **CCF anatomical region filtering:** For precise region filtering by Allen CCF anatomy (e.g. arcuate nucleus = ARC), use the CCF coordinates notebook: `alleninstitute.github.io/abc_atlas_access/notebooks/merfish_ccf_registration_tutorial.html`

5. **Restrict slides to a specific cluster:** Step 15 shows how to filter `cell_with_expr` by `subclass`/`cluster` before plotting, if you decide you want cluster-restricted slides later.

---
*Data from the Allen Brain Cell Atlas — Yao et al. 2023, Nature. CC BY NC 4.0 license.*